In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# 1. Vector Quantizer (single level)
# ============================================================
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        nn.init.uniform_(self.embeddings.weight, -1.0 / num_embeddings, 1.0 / num_embeddings)

    def forward(self, inputs):
        # inputs: [B, D]
        inputs_sq = inputs.pow(2).sum(dim=1, keepdim=True)           # [B, 1]
        emb_sq = self.embeddings.weight.pow(2).sum(dim=1)            # [K]
        cross = torch.matmul(inputs, self.embeddings.weight.t())     # [B, K]
        dist = inputs_sq + emb_sq.unsqueeze(0) - 2 * cross           # [B, K]
        
        indices = torch.argmin(dist, dim=1)
        quantized = self.embeddings(indices)
        
        # VQ losses
        codebook_loss = F.mse_loss(quantized, inputs.detach())
        commitment_loss = F.mse_loss(quantized.detach(), inputs)
        vq_loss = codebook_loss + self.commitment_cost * commitment_loss
        
        return quantized, indices, vq_loss


# ============================================================
# 2. Residual Quantizer (the core of RQ-VAE)
# ============================================================
class ResidualQuantizer(nn.Module):
    def __init__(self, num_quantizers, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_quantizers = num_quantizers
        self.quantizers = nn.ModuleList([
            VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
            for _ in range(num_quantizers)
        ])

    def forward(self, z):
        """
        Returns:
            quantized_st : straight-through version (use this for decoder)
            total_vq_loss
            all_indices  : list of index tensors (this is your semantic ID)
            quantized    : raw sum of codebook vectors (for inspection)
        """
        quantized = torch.zeros_like(z)
        current_residual = z.clone()
        all_indices = []
        vq_losses = []

        for quantizer in self.quantizers:
            q, indices, vq_loss = quantizer(current_residual)
            
            all_indices.append(indices)
            vq_losses.append(vq_loss)
            
            quantized = quantized + q
            current_residual = current_residual - q          # update residual

        total_vq_loss = torch.stack(vq_losses).mean()
        
        # Straight-through estimator on the TOTAL quantization
        quantized_st = z + (quantized - z).detach()
        
        return quantized_st, total_vq_loss, all_indices, quantized


# ============================================================
# 3. Full RQ-VAE (toy version)
# ============================================================
class RQVAE(nn.Module):
    def __init__(self, input_dim, latent_dim, num_quantizers, num_embeddings, commitment_cost=0.25):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        self.rq = ResidualQuantizer(num_quantizers, num_embeddings, latent_dim, commitment_cost)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        quantized_st, vq_loss, indices, quantized = self.rq(z)
        x_recon = self.decoder(quantized_st)
        return x_recon, vq_loss, indices, z, quantized

    @torch.no_grad()
    def encode(self, x):
        z = self.encoder(x)
        _, _, indices, _ = self.rq(z)
        return indices, z

    @torch.no_grad()
    def get_semantic_ids(self, x):
        indices, _ = self.encode(x)
        return indices   # list of tensors, one per level


# ============================================================
# 4. Dummy Data (clustered)
# ============================================================
torch.manual_seed(42)
n_samples = 1024
input_dim = 8
n_clusters = 8

centers = torch.randn(n_clusters, input_dim) * 2.0
cluster_ids = torch.randint(0, n_clusters, (n_samples,))
X = centers[cluster_ids] + torch.randn(n_samples, input_dim) * 0.25   # small noise

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)


# ============================================================
# 5. Training
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RQVAE(
    input_dim=input_dim,
    latent_dim=4,
    num_quantizers=2,        # try 1 vs 2 to see the effect of residual
    num_embeddings=8,
    commitment_cost=0.25
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training RQ-VAE...")
for epoch in range(1, 101):
    model.train()
    total_recon = total_vq = 0.0
    
    for batch in dataloader:
        x = batch[0].to(device)
        
        x_recon, vq_loss, indices, z, quantized = model(x)
        recon_loss = F.mse_loss(x_recon, x)
        loss = recon_loss + vq_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_recon += recon_loss.item()
        total_vq += vq_loss.item()
    
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Recon: {total_recon/len(dataloader):.4f} | VQ: {total_vq/len(dataloader):.4f}")


# ============================================================
# 6. Verification Utilities
# ============================================================
@torch.no_grad()
def check_code_usage(model, dataloader):
    model.eval()
    usage = [set() for _ in range(model.rq.num_quantizers)]
    for batch in dataloader:
        x = batch[0].to(device)
        indices, _ = model.encode(x)
        for lvl, idx in enumerate(indices):
            usage[lvl].update(idx.cpu().tolist())
    for lvl, s in enumerate(usage):
        print(f"Level {lvl}: {len(s)}/{model.rq.quantizers[lvl].num_embeddings} codes used")


@torch.no_grad()
def verify_manual_reconstruction(model, x_sample):
    """Check that summing codebook vectors manually gives same result"""
    model.eval()
    z = model.encoder(x_sample)
    quantized_st, _, indices, quantized = model.rq(z)
    
    manual_q = torch.zeros_like(z)
    for level, idx in enumerate(indices):
        cb = model.rq.quantizers[level].embeddings
        manual_q += cb(idx)
    
    max_diff = (manual_q - quantized).abs().max().item()
    print(f"Manual reconstruction max diff: {max_diff:.2e} (should be ~0)")
    return max_diff < 1e-6


# Run verification
print("\n=== Verification ===")
check_code_usage(model, dataloader)

x_sample = X[:8].to(device)
print("\nManual vs model quantized check:")
verify_manual_reconstruction(model, x_sample)

# Final reconstruction quality
with torch.no_grad():
    x_recon, _, _, _, _ = model(x_sample)
    print(f"Sample recon MSE: {F.mse_loss(x_recon, x_sample).item():.4f}")

Training RQ-VAE...
Epoch   1 | Recon: 4.5415 | VQ: 0.2984
Epoch  20 | Recon: 1.1209 | VQ: 7.1471
Epoch  40 | Recon: 0.4058 | VQ: 1.1959
Epoch  60 | Recon: 0.1314 | VQ: 0.8134
Epoch  80 | Recon: 0.0820 | VQ: 0.2116
Epoch 100 | Recon: 0.0668 | VQ: 0.1406

=== Verification ===
Level 0: 5/8 codes used
Level 1: 7/8 codes used

Manual vs model quantized check:
Manual reconstruction max diff: 0.00e+00 (should be ~0)
Sample recon MSE: 0.0651
